Cleaning solar permits dataset from https://catalog.data.gov/dataset/solar-permit-applications

In [1]:
import pandas as pd         #   library pandas

Read in the csv data and change data types to be accurate

In [ ]:
data_types = {'originaladdress2': str,              #   create a dictionary of specific columns and the data type they should be. originaladdress2, originalzip, contractorzip, and ownerzip all get read as numeric if their data types aren't assigned
              'originalzip': str, 
              'contractorzip': str,
              'ownerzip': str
              }
solar_permits = pd.read_csv('File Path',        #   reads in csv data from path
                            sep = ';',      #   this data is separated by semicolons instead of commas
                            parse_dates = ['issuedate','applieddate','statusdate'],      #   converts date columns to datetime data types
                            dtype = data_types        #   applies the data_types dictionary to the dataframe to change the data type of certain columns
                            )

Import the cleaning.py functions

In [3]:
import cleaning as clean        #   library the cleaning functions from cleaning.py

Apply zip_clean function to zip columns

In [ ]:
#   use the zip_clean function on the three zipcode columns (originalzip, contractor zip, ownerzip) in the solar_permits dataset
clean.zip_clean(solar_permits, 'originalzip')
clean.zip_clean(solar_permits, 'contractorzip')
clean.zip_clean(solar_permits, 'ownerzip')

Cleaning owner name and address columns (some names are in the address columns) using owner_address_clean

In [ ]:
solar_permits = clean.owner_address_clean(solar_permits, 'ownername', 'owneraddress1', 'owneraddress2')     #   uses the owner_address_clean function to clean the owner name and address data

Cleaning contractor address columns (some are out of order) using contractor_address_clean

In [ ]:
solar_permits = clean.contractor_address_clean(solar_permits, 'contractoraddress1', 'contractoraddress2')       #   uses the contractor_address_clean function to clean the contractor address data

Import the excel.py functions

In [8]:
import excel        #   library the excel functions from excel.py

Append all new contractor information to the 'Contractor_Names.xlsx' workbook. After running the following code block, use conditional formatting and filtering in excel to determine which names need to be updated in the 'contractorname_clean' column

In [ ]:
excel.excel_append_contractors(solar_permits, 'contractorcompanyname', 'contractortrade', 'contractoraddress1', 'contractoraddress2', 'contractorphone', 'Contractor_Names.xlsx', 'Contractor Data')      #   uses the contractor_excel_append function to append new contractor data to the 'Contractor Data' sheet in the 'Contractor_Names.xlsx' workbook

Add a protected copy of the cleaned contractor data and add it to a hidden meta data sheet

In [9]:
excel.create_protection_sheet('Contractor_Names.xlsx', 'Contractor Data')       #   uses the create_protection_sheet function to create a protection sheet of the contractor data after contractor name data was manually validated and corrected in Excel

Bring in the cleaned contractor names and replace the old contractor names data with the cleaned data

In [ ]:
solar_permits = excel.clean_contractor_names(solar_permits, 'Contractor_Names.xlsx', 'contractorcompanyname', 'contractorname_clean')

Import the address_validation.py functions

In [11]:
import address_validation as validate       #   library address_validationfunctions from address_validation.py

Validate address data using Google Maps API

In [ ]:
gmaps_key = 'Google Maps API Key'       #   google maps api key as a string

solar_permits_validated = validate.apply_address_validation(solar_permits, 'originaladdress1', 'originalcity', gmaps_key)        #   uses the apply_address_validation function to validate all addresses in the dataset. as new data gets released set 'existing_data' to the previous output and 'primary_key' to 'permitnum' (took 4m 43.8s for 1980 rows)

Import the inflation_adjust.py functions

In [ ]:
import inflation_adjust.py as inf

Adjust 'projectcost' for inflation

In [ ]:
url = 'https://data.bls.gov/timeseries/CUUR0000SA0?years_option=all_years'      #   Bureau of Labor Statistics website with consumer price index data

solar_permits_inflation = inf.calculate_inflation_adjustment(solar_permits_validated, 'applieddate', 'applieddate', url)        #   uses the calculate_inflation_adjustment function to adjust the 'projectcost' data based on when the application was applied ('applieddate')

Save the cleaned data to a csv

In [ ]:
solar_permits_inflation.to_csv('solar_permits_clean.csv', index = False, na_rep = '')        #   save the validated dataset after apply_address_validation has been ran so the data can be used as new data gets released